RateLimiter caught an error, retrying (0/2 tries). Called with (*('Via G. Revere 6, MILANO, ITALY',), **{}).
Traceback (most recent call last):
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 1374, in getresponse
    response.begin()
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "

KeyboardInterrupt: 

In [2]:
import glob
import os

# Directory where temporary batch outputs are stored
cleanup_dir = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs"

# Delete all previous .geojson batch files
for f in glob.glob(os.path.join(cleanup_dir, "*.geojson")):
    try:
        os.remove(f)
        print(f"✅ Deleted: {f}")
    except Exception as e:
        print(f"⚠️ Error deleting {f}: {e}")


In [1]:
import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from shapely.geometry import Point
import os
import time

# === CONFIGURATION ===
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
output_folder = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs"
os.makedirs(output_folder, exist_ok=True)

batch_size = 500
start_row = 0  # change this if resuming from a specific row
geolocator = Nominatim(user_agent="epc_geocoder", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=10.0)

# === LOAD EPC DATA ===
df = pd.read_csv(epc_path, low_memory=False)
df = df[df["COMUNE"].str.upper() == "MILANO"].copy()
df["INDIRIZZO_NORM"] = df["INDIRIZZO"].str.upper().str.strip()
df["full_address"] = df["INDIRIZZO_NORM"] + ", MILANO, ITALY"
df = df.reset_index(drop=True)

# === PROCESS IN BATCHES ===
total = len(df)
for batch_start in range(start_row, total, batch_size):
    batch_end = min(batch_start + batch_size, total)
    batch_df = df.iloc[batch_start:batch_end].copy()

    print(f"\n🔄 Processing rows {batch_start} to {batch_end}...")

    results = []
    for i, address in enumerate(batch_df["full_address"]):
        try:
            location = geocode(address)
            if location:
                results.append((location.latitude, location.longitude))
            else:
                results.append((None, None))
        except Exception as e:
            print(f"⚠️ Error at row {batch_start + i}: {e}")
            results.append((None, None))

    batch_df[["latitude", "longitude"]] = pd.DataFrame(results, index=batch_df.index)
    batch_df = batch_df.dropna(subset=["latitude", "longitude"])

    if not batch_df.empty:
        gdf = gpd.GeoDataFrame(
            batch_df,
            geometry=[Point(xy) for xy in zip(batch_df.longitude, batch_df.latitude)],
            crs="EPSG:4326"
        )

        out_path = os.path.join(output_folder, f"epc_geocoded_batch_{batch_end}.geojson")
        gdf.to_file(out_path, driver="GeoJSON")
        print(f"✅ Saved: {out_path}")
    else:
        print("⚠️ No successful geocodes in this batch.")

    # Optional sleep between batches
    time.sleep(3)



🔄 Processing rows 0 to 500...


KeyboardInterrupt: 

In [2]:
import os
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

# Load EPC dataset
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
epc_df = pd.read_csv(epc_path, low_memory=False)

# Prepare full address
epc_df["full_address"] = epc_df["INDIRIZZO"].astype(str) + ", " + epc_df["COMUNE"].astype(str) + ", ITALY"

# Output folder
output_folder = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\geo_outputs"
os.makedirs(output_folder, exist_ok=True)

# Detect completed batches
existing_batches = [int(f.split('_')[-1].replace('.csv','')) for f in os.listdir(output_folder) if f.startswith("epc_geocoded_batch_")]
start_index = max(existing_batches) + 1 if existing_batches else 0

# Set up geocoder
geolocator = Nominatim(user_agent="epc_milano_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, error_wait_seconds=10.0, max_retries=3)

batch_size = 500
total = len(epc_df)

print(f"📌 Resuming from batch {start_index} — total rows: {total}")

for i in range(start_index, (total // batch_size) + 1):
    batch_start = i * batch_size
    batch_end = min((i + 1) * batch_size, total)
    
    batch = epc_df.iloc[batch_start:batch_end].copy()
    print(f"🔄 Processing rows {batch_start} to {batch_end}...")

    results = []
    for addr in batch["full_address"]:
        try:
            location = geocode(addr)
            if location:
                results.append((addr, location.latitude, location.longitude))
            else:
                results.append((addr, None, None))
        except Exception as e:
            print(f"❌ Error geocoding {addr}: {e}")
            results.append((addr, None, None))

    result_df = pd.DataFrame(results, columns=["full_address", "latitude", "longitude"])
    output_path = os.path.join(output_folder, f"epc_geocoded_batch_{batch_start}.csv")
    result_df.to_csv(output_path, index=False)
    print(f"✅ Saved: {output_path}")

    # Optional sleep between batches
    time.sleep(2)


📌 Resuming from batch 0 — total rows: 342684
🔄 Processing rows 0 to 500...


RateLimiter caught an error, retrying (0/3 tries). Called with (*('VIA ALZAIA NAVIGLIO PAVESE 260, MILANO, ITALY',), **{}).
Traceback (most recent call last):
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 1374, in getresponse
    response.begin()
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^

KeyboardInterrupt: 

In [3]:
import pandas as pd
import os
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

# === Settings ===
input_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
save_dir = "geo_outputs"
batch_size = 500  # You can change to 250 or 1000 if needed

# === Ensure output folder exists ===
os.makedirs(save_dir, exist_ok=True)

# === Load source file ===
df = pd.read_csv(input_path, low_memory=False)
df = df[df["COMUNE"].str.upper() == "MILANO"]  # Filter to Milan only

# === Setup geocoder ===
geolocator = Nominatim(user_agent="epc_geocoder", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=3)

# === Determine starting batch ===
existing_batches = sorted([
    int(f.split("_")[-1].split(".")[0]) 
    for f in os.listdir(save_dir) if f.startswith("epc_geocoded_batch_")
])
start_batch = max(existing_batches) + 1 if existing_batches else 0
start_index = start_batch * batch_size

print(f"📌 Resuming from batch {start_batch} — total rows: {len(df)}")

# === Geocode loop ===
for batch_start in range(start_index, len(df), batch_size):
    batch_end = min(batch_start + batch_size, len(df))
    batch = df.iloc[batch_start:batch_end].copy()

    print(f"🔄 Processing rows {batch_start} to {batch_end}...")

    # Construct full address
    batch["full_address"] = batch["INDIRIZZO"].astype(str) + ", " + batch["COMUNE"].astype(str) + ", ITALY"

    # Geocode each row
    lats, lons = [], []
    for address in batch["full_address"]:
        try:
            location = geocode(address)
            lats.append(location.latitude if location else None)
            lons.append(location.longitude if location else None)
        except Exception as e:
            lats.append(None)
            lons.append(None)

    batch["latitude"] = lats
    batch["longitude"] = lons

    # Save batch
    out_path = os.path.join(save_dir, f"epc_geocoded_batch_{str(batch_start).zfill(3)}.csv")
    batch.to_csv(out_path, index=False)
    print(f"✅ Saved batch to: {out_path}")


📌 Resuming from batch 0 — total rows: 342684
🔄 Processing rows 0 to 500...


KeyboardInterrupt: 

In [4]:
import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from shapely.geometry import Point
import os
import time
import re

# ----------------------------------
# 1. Load the full EPC dataset
# ----------------------------------
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
df = pd.read_csv(epc_path, low_memory=False)

# ----------------------------------
# 2. Prepare address column
# ----------------------------------
df = df[df["COMUNE"].astype(str).str.upper().str.strip() == "MILANO"].copy()
df["full_address"] = df["INDIRIZZO"].astype(str).str.strip() + ", MILANO, ITALY"

# ----------------------------------
# 3. Setup batch saving directory
# ----------------------------------
save_dir = "geo_outputs"
os.makedirs(save_dir, exist_ok=True)

# ----------------------------------
# 4. Detect existing batches and resume correctly
# ----------------------------------
batch_size = 500
existing_batches = []
for f in os.listdir(save_dir):
    match = re.search(r'epc_geocoded_batch_(\d+)\.csv', f)
    if match:
        existing_batches.append(int(match.group(1)))
start_batch = max(existing_batches) // batch_size + 1 if existing_batches else 0
start_index = start_batch * batch_size
print(f"📌 Resuming from batch {start_batch} — total rows: {len(df)}")

# ----------------------------------
# 5. Setup geocoder (Nominatim)
# ----------------------------------
geolocator = Nominatim(user_agent="milano_epc_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=10)

# ----------------------------------
# 6. Batch loop (run safely, save frequently)
# ----------------------------------
for i in range(start_index, len(df), batch_size):
    batch = df.iloc[i:i + batch_size].copy()
    print(f"🔄 Processing rows {i} to {i + len(batch)}...")

    # Geocode each address
    batch["location"] = batch["full_address"].apply(geocode)
    batch["latitude"] = batch["location"].apply(lambda loc: loc.latitude if loc else None)
    batch["longitude"] = batch["location"].apply(lambda loc: loc.longitude if loc else None)
    batch = batch.drop(columns=["location"])

    # Convert to GeoDataFrame
    batch = batch.dropna(subset=["latitude", "longitude"])
    if len(batch) == 0:
        print("⚠️ No valid geocodes in this batch.")
        continue

    gdf = gpd.GeoDataFrame(
        batch,
        geometry=[Point(xy) for xy in zip(batch["longitude"], batch["latitude"])],
        crs="EPSG:4326"
    )

    # Save this batch
    batch_file = os.path.join(save_dir, f"epc_geocoded_batch_{i}.csv")
    gdf.to_csv(batch_file, index=False)
    print(f"✅ Saved {len(gdf)} geocoded entries to: {batch_file}")

    # OPTIONAL: Pause a bit to avoid API throttling
    time.sleep(2)


📌 Resuming from batch 0 — total rows: 342684
🔄 Processing rows 0 to 500...


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Via G. Silva 43, MILANO, ITALY',), **{}).
Traceback (most recent call last):
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
TimeoutError: timed out

The above exception was the direct cause of the following exception:

Traceback (mos

KeyboardInterrupt: 

In [5]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# === CONFIGURATION ===
epc_path = "Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
output_dir = "geo_outputs"
batch_size = 500

# === PREP ===
os.makedirs(output_dir, exist_ok=True)
epc_df = pd.read_csv(epc_path, low_memory=False)
epc_df['full_address'] = epc_df['INDIRIZZO'].astype(str).str.upper() + ", MILANO, ITALY"
total_rows = len(epc_df)

# === RESUME POINT ===
saved_batches = [int(f.split('_')[-1].split('.')[0]) for f in os.listdir(output_dir) if f.startswith("epc_batch_")]
start_batch = max(saved_batches) + 1 if saved_batches else 0
start_row = start_batch * batch_size

print(f"\n\U0001F4CC Resuming from batch {start_batch} — total rows: {total_rows}")

# === GEOCODER SETUP ===
geolocator = Nominatim(user_agent="milano_epc_geocoder", timeout=10)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=2.5,
    max_retries=3,
    error_wait_seconds=15,
    swallow_exceptions=True
)

# === BATCH GEOCODING LOOP ===
for i in range(start_row, total_rows, batch_size):
    end_i = min(i + batch_size, total_rows)
    print(f"\U0001F504 Processing rows {i} to {end_i}...")

    batch_df = epc_df.iloc[i:end_i].copy()
    batch_df['location'] = batch_df['full_address'].apply(geocode)
    batch_df['latitude'] = batch_df['location'].apply(lambda loc: loc.latitude if loc else None)
    batch_df['longitude'] = batch_df['location'].apply(lambda loc: loc.longitude if loc else None)
    batch_df['geometry'] = batch_df.apply(
        lambda row: Point(row['longitude'], row['latitude']) if pd.notnull(row['latitude']) and pd.notnull(row['longitude']) else None,
        axis=1
    )

    gdf = gpd.GeoDataFrame(batch_df, geometry='geometry', crs="EPSG:4326")
    out_path = os.path.join(output_dir, f"epc_batch_{i//batch_size}.geojson")
    gdf.to_file(out_path, driver="GeoJSON")
    print(f"\u2705 Saved batch {i//batch_size} to {out_path}\n")

print("\n✅ All done.")


FileNotFoundError: [Errno 2] No such file or directory: 'Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv'

In [6]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from datetime import datetime

# === PATH SETUP ===
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
output_dir = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\output_epc_batches"
os.makedirs(output_dir, exist_ok=True)

# === GEOCODER SETUP ===
geolocator = Nominatim(user_agent="epc_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=3, error_wait_seconds=10.0)

# === LOAD DATA ===
epc_df = pd.read_csv(epc_path, low_memory=False)
epc_df['full_address'] = epc_df['INDIRIZZO'].astype(str).str.upper() + ", MILANO, ITALY"
total_rows = len(epc_df)

# === CONFIG ===
batch_size = 500
max_batches = total_rows // batch_size + 1

# === FIND LAST PROCESSED BATCH ===
existing_batches = [
    int(f.split("_")[2].split(".")[0]) 
    for f in os.listdir(output_dir) 
    if f.startswith("epc_batch_") and f.endswith(".geojson")
]
start_batch = max(existing_batches) + 1 if existing_batches else 0
print(f"📌 Resuming from batch {start_batch} — total rows: {total_rows}")

# === PROCESS LOOP ===
for batch_num in range(start_batch, max_batches):
    start_idx = batch_num * batch_size
    end_idx = min((batch_num + 1) * batch_size, total_rows)
    batch_df = epc_df.iloc[start_idx:end_idx].copy()
    print(f"\n🔄 Processing rows {start_idx} to {end_idx}...")

    latitudes = []
    longitudes = []

    for addr in batch_df['full_address']:
        try:
            location = geocode(addr)
            if location:
                latitudes.append(location.latitude)
                longitudes.append(location.longitude)
            else:
                latitudes.append(None)
                longitudes.append(None)
        except Exception as e:
            latitudes.append(None)
            longitudes.append(None)
            print(f"⚠️ Geocoding failed for: {addr}\n{e}")

    # Attach coordinates
    batch_df['latitude'] = latitudes
    batch_df['longitude'] = longitudes

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(
        batch_df,
        geometry=[Point(xy) if pd.notnull(xy[0]) and pd.notnull(xy[1]) else None for xy in zip(batch_df['longitude'], batch_df['latitude'])],
        crs="EPSG:4326"
    )

    # Save batch
    output_file = os.path.join(output_dir, f"epc_batch_{batch_num}.geojson")
    gdf.to_file(output_file, driver="GeoJSON")
    print(f"✅ Saved batch {batch_num} to {output_file}")


📌 Resuming from batch 0 — total rows: 342684

🔄 Processing rows 0 to 500...


RateLimiter caught an error, retrying (0/3 tries). Called with (*('VIA G. SILVA 43, MILANO, ITALY',), **{}).
Traceback (most recent call last):
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 1374, in getresponse
    response.begin()
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "

KeyboardInterrupt: 

In [7]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os
import re

# === INPUT & OUTPUT SETTINGS ===
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
output_dir = r"F:\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs"
os.makedirs(output_dir, exist_ok=True)

# === READ EPC DATA ===
epc_df = pd.read_csv(epc_path, low_memory=False)
epc_df['full_address'] = epc_df['INDIRIZZO'].astype(str).str.upper() + ", MILANO, ITALY"
total_rows = len(epc_df)

# === DETECT LAST SAVED BATCH ===
saved_files = os.listdir(output_dir)
batch_numbers = []
for file in saved_files:
    match = re.search(r'batch_(\d+)', file)
    if match:
        batch_numbers.append(int(match.group(1)))

start_batch = max(batch_numbers) if batch_numbers else 0

print(f"\n📌 Resuming from batch {start_batch} — total rows: {total_rows}")

# === GEOCODER SETUP ===
geolocator = Nominatim(user_agent="epc_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=3, error_wait_seconds=10)

# === BATCH PROCESSING ===
batch_size = 500
for batch_start in range(start_batch, total_rows, batch_size):
    batch_end = min(batch_start + batch_size, total_rows)
    print(f"\n🔄 Processing rows {batch_start} to {batch_end}...")

    batch_df = epc_df.iloc[batch_start:batch_end].copy()
    batch_df['location'] = batch_df['full_address'].apply(lambda addr: geocode(addr))
    batch_df['latitude'] = batch_df['location'].apply(lambda loc: loc.latitude if loc else None)
    batch_df['longitude'] = batch_df['location'].apply(lambda loc: loc.longitude if loc else None)

    valid = batch_df.dropna(subset=['latitude', 'longitude'])
    if not valid.empty:
        gdf = gpd.GeoDataFrame(
            valid,
            geometry=[Point(xy) for xy in zip(valid.longitude, valid.latitude)],
            crs="EPSG:4326"
        )
        out_path = os.path.join(output_dir, f"epc_geocoded_batch_{batch_end}.geojson")
        gdf.to_file(out_path, driver="GeoJSON")
        print(f"✅ Saved: {out_path}")
    else:
        print("⚠️  No successful geocodes in this batch.")
        


📌 Resuming from batch 0 — total rows: 342684

🔄 Processing rows 0 to 500...


RateLimiter caught an error, retrying (0/3 tries). Called with (*('VIA CALDIROLA 6, MILANO, ITALY',), **{}).
Traceback (most recent call last):
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 1374, in getresponse
    response.begin()
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "

KeyboardInterrupt: 

In [1]:
import os
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# === CONFIGURATION ===
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
output_dir = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs"
batch_size = 500

# === SETUP ===
os.makedirs(output_dir, exist_ok=True)
epc_df = pd.read_csv(epc_path, low_memory=False)
epc_df['full_address'] = epc_df['INDIRIZZO'].astype(str).str.upper() + ", MILANO, ITALY"
total_rows = len(epc_df)

# === Detect Already Processed Batches ===
existing_batches = [
    int(f.split("_")[-1].replace(".geojson", ""))
    for f in os.listdir(output_dir)
    if f.startswith("epc_geocoded_batch_") and f.endswith(".geojson")
]
last_done = max(existing_batches) if existing_batches else 0
print(f"📌 Resuming from batch {last_done} — total rows: {total_rows}")

# === Geocoder Setup ===
geolocator = Nominatim(user_agent="epc_geocoder", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=3, error_wait_seconds=10)

# === Processing Loop ===
for start in range(last_done, total_rows, batch_size):
    end = min(start + batch_size, total_rows)
    print(f"\n🔄 Processing rows {start} to {end}...")

    batch = epc_df.iloc[start:end].copy()
    batch["location"] = batch["full_address"].apply(lambda addr: geocode(addr) if pd.notnull(addr) else None)
    batch["latitude"] = batch["location"].apply(lambda loc: loc.latitude if loc else None)
    batch["longitude"] = batch["location"].apply(lambda loc: loc.longitude if loc else None)

    # Drop any rows without successful geocoding
    batch = batch.dropna(subset=["latitude", "longitude"])
    if batch.empty:
        print("⚠️ No valid results in this batch.")
        continue

    # Save batch as GeoJSON
    gdf = gpd.GeoDataFrame(
        batch,
        geometry=[Point(xy) for xy in zip(batch.longitude, batch.latitude)],
        crs="EPSG:4326" 
    )
    out_path = os.path.join(output_dir, f"epc_geocoded_batch_{end}.geojson")
    gdf.to_file(out_path, driver="GeoJSON")
    print(f"✅ Saved: {out_path}") 
    

📌 Resuming from batch 317000 — total rows: 342684

🔄 Processing rows 317000 to 317500...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_317500.geojson

🔄 Processing rows 317500 to 318000...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_318000.geojson

🔄 Processing rows 318000 to 318500...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_318500.geojson

🔄 Processing rows 318500 to 319000...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_319000.geojson

🔄 Processing rows 319000 to 319500...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collectio

RateLimiter caught an error, retrying (0/3 tries). Called with (*('VIA SOLFERINO 23, MILANO, ITALY',), **{}).
Traceback (most recent call last):
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
TimeoutError: timed out

The above exception was the direct cause of the following exception:

Traceback (mo

✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_323000.geojson

🔄 Processing rows 323000 to 323500...


RateLimiter caught an error, retrying (0/3 tries). Called with (*('VIALE CERTOSA 30, MILANO, ITALY',), **{}).
Traceback (most recent call last):
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
TimeoutError: timed out

The above exception was the direct cause of the following exception:

Traceback (mo

✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_323500.geojson

🔄 Processing rows 323500 to 324000...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_324000.geojson

🔄 Processing rows 324000 to 324500...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_324500.geojson

🔄 Processing rows 324500 to 325000...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_325000.geojson

🔄 Processing rows 325000 to 325500...
✅ Saved: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\geo_outputs\epc_geocoded_batch_325500.geojson

🔄 Processing rows 325500 to 32600